## 0. Install dependencies.

In [ ]:
!pip install gensim scikit-learn imbalanced-learn nltk pandas numpy matplotlib openpyxl
!pip install scispacy negspacy spacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [ ]:
import warnings, re, string
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ["stopwords", "punkt", "punkt_tab"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
from nltk.corpus import stopwords

import gensim
from gensim import corpora
from gensim.models import LdaModel, TfidfModel, CoherenceModel

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_score, recall_score, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("gensim", gensim.__version__)

## 1. Upload and load your dataset

Provide two files:

- **Your EHR export** (CSV or Excel), one row per procedure record. Expected columns:
  `record_id`, `redcap_repeat_instrument`, `redcap_repeat_instance`, `encounter_id`, `Admission Date`, `Discharge Date`, `primary diagnosis`, `secondary diagnosis`, `procedure report`.
  The **`encounter_id`** column is the hospitalization key — several procedure rows (different `redcap_repeat_instance`) roll up to one `encounter_id`. `redcap_repeat_instance` is just a per-patient running counter and is **not** used for grouping.
- The **AHRQ CCSR mapping** `code_to_CCSR.csv` (columns `Code`, `Category`), used to convert ICD-10-CM codes to text.

In [ ]:
# ---- File paths (edit these) ----
DATA_PATH = "your_dataset.csv"      # <-- your uploaded EHR export (.csv or .xlsx)
CCSR_PATH = "code_to_CCSR.csv"      # <-- AHRQ CCSR mapping (Code, Category)

# ---- Column names in your file (edit if they differ) ----
COL_RECORD_ID    = "record_id"
COL_INSTRUMENT   = "redcap_repeat_instrument"
COL_INSTANCE     = "redcap_repeat_instance"
COL_ENCOUNTER    = "encounter_id"          # the hospitalization key (groups rows)
COL_ADMIT        = "Admission Date"
COL_DISCHARGE    = "Discharge Date"
COL_PRIMARY_DX   = "primary diagnosis"
COL_SECONDARY_DX = "secondary diagnosis"
COL_PROC         = "procedure report"

# Set False if your diagnosis columns already contain TEXT (not ICD-10-CM codes)
USE_CCSR = True

In [ ]:
def load_table(path):
    return pd.read_excel(path) if str(path).lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)

data = load_table(DATA_PATH)
data.columns = [c.strip() for c in data.columns]

required = [COL_RECORD_ID, COL_INSTANCE, COL_ENCOUNTER, COL_ADMIT, COL_DISCHARGE,
            COL_PRIMARY_DX, COL_SECONDARY_DX, COL_PROC]
missing = [c for c in required if c not in data.columns]
assert not missing, f"Missing expected columns: {missing}\nFound: {list(data.columns)}"

# encounter = one hospitalization, identified by the explicit encounter_id column.
# (Several procedure rows with different redcap_repeat_instance share one encounter_id.)
data["encounter_id"] = data[COL_ENCOUNTER].astype(str)
print(f"{len(data)} rows | {data['encounter_id'].nunique()} encounters | "
      f"{data[COL_RECORD_ID].nunique()} patients")
data.head(3)

## 2. Diagnostic codes → CCSR text

Primary + secondary diagnosis codes are combined per encounter. **Both columns may contain multiple codes** separated by `;` (or `,`/`|`/`/`); `split_codes` handles that. Each code has its **decimal removed** (`I50.9 → I509`), is mapped to its **CCSR category name(s)** via the uploaded file (a code may map to several), and the `Heart failure` category is dropped (every patient has it). Set `USE_CCSR=False` above if your columns already hold text.

In [ ]:
# Load CCSR mapping: decimal-free code -> list of category names
ccsr = pd.read_csv(CCSR_PATH)
ccsr.columns = [c.strip() for c in ccsr.columns]
ccsr["Code"] = ccsr["Code"].astype(str).str.strip()
ccsr["Category"] = ccsr["Category"].astype(str).str.strip()
code2cats = ccsr.groupby("Code")["Category"].apply(list).to_dict()
print(f"CCSR: {ccsr.Code.nunique()} codes -> {ccsr.Category.nunique()} categories")

HF_CATEGORY = "Heart failure"

def strip_decimal(code):
    return str(code).replace(".", "").strip().upper()

def split_codes(val):
    if pd.isna(val): return []
    return [c.strip() for c in re.split(r"[;,|/]+|\s{2,}", str(val)) if c.strip()]

def codes_to_text(codes, drop=frozenset([HF_CATEGORY])):
    cats = []
    for c in codes:
        for cat in code2cats.get(strip_decimal(c), []):
            if cat not in drop:
                cats.append(cat)
    return "; ".join(cats)

In [ ]:
# Combine primary + secondary codes per encounter (union, de-duplicated)
def row_codes(r):
    return split_codes(r[COL_PRIMARY_DX]) + split_codes(r[COL_SECONDARY_DX])

data["_codes"] = data.apply(row_codes, axis=1)
enc_codes = (data.groupby("encounter_id")["_codes"]
             .apply(lambda lists: list(dict.fromkeys(c for l in lists for c in l))))

STOP = set(stopwords.words("english"))
def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(t for t in text.split() if t not in STOP and len(t) > 2)

if USE_CCSR:
    dx_text = enc_codes.apply(codes_to_text)
else:
    # diagnosis columns already contain text
    dx_text = (data.groupby("encounter_id")
               .apply(lambda g: "; ".join(g[COL_PRIMARY_DX].fillna("").astype(str)) + "; " +
                                "; ".join(g[COL_SECONDARY_DX].fillna("").astype(str))))

diag = pd.DataFrame({"encounter_id": dx_text.index,
                     "dx_text": dx_text.values})
diag["dx_tokens"] = diag["dx_text"].apply(basic_clean)
diag = diag[diag["dx_tokens"].str.strip() != ""].reset_index(drop=True)
print(f"{len(diag)} encounters with diagnostic text")
diag.head(3)

## 3. Procedure reports → extract & clean the IMPRESSION

Pull the **IMPRESSION** section from each `procedure report`, lowercase, strip digits and punctuation — but turn commas/semicolons/periods into **sentence boundaries** so negspaCy scopes "no" to the correct finding instead of bleeding across a comma-separated list.

In [ ]:
# Extract Impression attribute (handles 'IMPRESSION:' and 'IMPRESSION :')
def extract_impression(text):
    if pd.isna(text):
        return None
    match = re.search(r'impression\s*:\s*(.*)', str(text), flags=re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

# Clean: keep comma/period as SENTENCE boundaries
def clean_text(text):
    if pd.isna(text):
        return None
    text = str(text).lower()
    text = re.sub(r'[,;.]+', ' . ', text)
    other_punct = ''.join(ch for ch in string.punctuation if ch != '.')
    text = re.sub(r'[\d' + re.escape(other_punct) + ']', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

proc = data[["encounter_id", COL_PROC]].copy()
proc["impression_extracted"] = proc[COL_PROC].apply(extract_impression)
proc["impression_cleaned"]   = proc["impression_extracted"].apply(clean_text)
proc = proc.dropna(subset=["impression_cleaned"])
proc = proc[proc["impression_cleaned"].str.strip() != ""].reset_index(drop=True)
print(f"{len(proc)} procedure reports with an IMPRESSION")
proc[["impression_extracted", "impression_cleaned"]].head()

## 4. Negation detection + retain entity phrases (scispaCy + negspaCy)

Load `en_core_sci_sm`, build a clinical negation termset, and add negspaCy's `negex`. For every entity scispaCy recognizes we keep the **phrase containing that entity** (underscore-joined), prefixing negated ones with `no_`.

*Optional:* to make sure specific known findings are never missed/clipped by the statistical model, list them in `FINDINGS_LEXICON` and a spaCy `EntityRuler` will capture them deterministically. Leave it empty to use scispaCy entities only.

In [ ]:
# Perform Negation Detection
import scispacy
import spacy
from negspacy.negation import Negex
from negspacy.termsets import termset

nlp = spacy.load("en_core_sci_sm")

ts = termset("en_clinical")
ts.add_patterns({
    "preceding_negations": ["unable"],
    "following_negations": ["was negative"]
})

# OPTIONAL deterministic capture of known findings (leave [] to rely on scispaCy NER)
FINDINGS_LEXICON = []   # e.g. ["t wave abnormality", "lv dysfunction", "pleural effusion"]
if FINDINGS_LEXICON:
    ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
    ruler.add_patterns([{"label": "FINDING", "pattern": p.lower()} for p in FINDINGS_LEXICON])

if "negex" not in nlp.pipe_names:
    nlp.add_pipe("negex", config={"neg_termset": ts.get_patterns()})

def process_impression(text):
    if pd.isna(text):
        return None
    doc = nlp(str(text).lower())
    phrases = []
    for ent in doc.ents:                      # retain phrases containing a recognized entity
        ent_text = ent.text.strip().replace(" ", "_")
        phrases.append(f"no_{ent_text}" if ent._.negex else ent_text)
    return " ".join(phrases) if phrases else None

proc["impression_negex"] = proc["impression_cleaned"].apply(process_impression)
proc[["impression_cleaned", "impression_negex"]].head()

## 5. Stop-word removal, then merge per encounter

Remove generic radiology/boilerplate words, then combine all procedure reports of the same encounter and de-duplicate phrases (one `proc_text` per encounter).

In [ ]:
stop_words = set(stopwords.words('english'))
stop_words.update([
    'personalname', 'alphanumericid', 'examination', 'preliminary', 'impression',
    'chest', 'view', 'change', 'final', 'available', 'evidence', 'see', 'right',
    'stable', 'date', 'compare', 'interpretation', 'portable', 'lung', 'leave',
    'report', 'process', 'review', 'resident', 'radiologist', 'attend',
    'finding', 'findings', 'electronic', 'and', 'sign'
])

def remove_stopwords_custom(text):
    if pd.isna(text):
        return ''
    return ' '.join(t for t in str(text).lower().split() if t not in stop_words)

proc["proc_clean"] = proc["impression_negex"].apply(remove_stopwords_custom)
proc = proc[proc["proc_clean"].str.strip() != ""].reset_index(drop=True)

def dedup_join(series):
    seen, out = set(), []
    for txt in series:
        for tok in str(txt).split():
            if tok not in seen:
                seen.add(tok); out.append(tok)
    return " ".join(out)

proc_by_enc = (proc.groupby("encounter_id")["proc_clean"]
               .apply(dedup_join).reset_index()
               .rename(columns={"proc_clean": "proc_text"}))
print(f"{len(proc_by_enc)} encounters with procedure text")
proc_by_enc.head(3)

In [ ]:
# Per-encounter metadata: record_id + admission/discharge dates
dates = data.copy()
dates[COL_ADMIT]     = pd.to_datetime(dates[COL_ADMIT], errors="coerce")
dates[COL_DISCHARGE] = pd.to_datetime(dates[COL_DISCHARGE], errors="coerce")
enc_meta = (dates.groupby("encounter_id")
            .agg(record_id=(COL_RECORD_ID, "first"),
                 admission=(COL_ADMIT, "min"),
                 discharge=(COL_DISCHARGE, "max"))
            .reset_index())

# Final analysis set: encounters having diagnostic AND procedure text AND valid dates
records = (diag.merge(proc_by_enc, on="encounter_id", how="inner")
               .merge(enc_meta, on="encounter_id", how="inner"))
records = records.dropna(subset=["admission", "discharge"]).reset_index(drop=True)
print(f"Final analysis set: {len(records)} encounters")
records[["encounter_id", "record_id", "admission", "discharge", "dx_tokens", "proc_text"]].head(3)

## 6. Length of stay: days from dates, then 5 classes

| class | label | days |
|---|---|---|
| 0 | very short-term | 0–1 |
| 1 | short-term | 2–7 |
| 2 | medium-term | 8–14 |
| 3 | long-term | 15–21 |
| 4 | very long-term | >21 |

In [ ]:
records["los_days"] = (records["discharge"] - records["admission"]).dt.days
records = records[records["los_days"] >= 0].reset_index(drop=True)

def los_category(d):
    if d <= 1:  return 0
    if d <= 7:  return 1
    if d <= 14: return 2
    if d <= 21: return 3
    return 4

LOS_NAMES = ["very short (0-1d)", "short (2-7d)", "medium (8-14d)",
             "long (15-21d)", "very long (>21d)"]
records["los_class"] = records["los_days"].apply(los_category)
counts = records["los_class"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(range(5), [counts.get(i, 0) for i in range(5)], color="#4C72B0", edgecolor="black")
plt.xticks(range(5), [str(i + 1) for i in range(5)])
plt.xlabel("Length of stay class"); plt.ylabel("Number of records")
plt.title("Distribution of records by length-of-stay class (cf. Fig. 2)")
plt.tight_layout(); plt.show()
print(records["los_days"].describe()[["min", "mean", "max"]])
print(counts.rename(index=lambda i: LOS_NAMES[i]))

## 7. Topic modeling (LDA on TF-IDF) with coherence-selected topic counts

Two LDA models — diagnostic codes and procedure reports — both fit on a **TF-IDF** corpus (no bag-of-words). For **each**, we sweep the number of topics and pick the value with the **highest c_v coherence**.

In [ ]:
def build_corpus(texts):
    tokenized = [t.split() for t in texts]
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=2, no_above=0.6)
    bow = [dictionary.doc2bow(doc) for doc in tokenized]
    tfidf = TfidfModel(bow)
    tfidf_corpus = [tfidf[d] for d in bow]
    return tokenized, dictionary, tfidf_corpus

dx_tok, dx_dict, dx_tfidf = build_corpus(records["dx_tokens"])
pr_tok, pr_dict, pr_tfidf = build_corpus(records["proc_text"])
print("dx vocab:", len(dx_dict), "| proc vocab:", len(pr_dict))

In [ ]:
# Sweep number of topics and pick the highest-coherence k (on TF-IDF corpus)
K_VALUES = list(range(2, 21, 2))   # adjust range/step as needed

def select_k_by_coherence(tfidf_corpus, dictionary, tokenized, k_values):
    scores = []
    for k in k_values:
        lda = LdaModel(corpus=tfidf_corpus, id2word=dictionary, num_topics=k,
                       random_state=RANDOM_STATE, passes=5, alpha="auto")
        cm = CoherenceModel(model=lda, texts=tokenized, dictionary=dictionary, coherence="c_v")
        scores.append(cm.get_coherence())
    best_k = k_values[int(np.argmax(scores))]
    return best_k, scores

dx_best_k, dx_scores = select_k_by_coherence(dx_tfidf, dx_dict, dx_tok, K_VALUES)
pr_best_k, pr_scores = select_k_by_coherence(pr_tfidf, pr_dict, pr_tok, K_VALUES)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(K_VALUES, dx_scores, marker="o"); ax[0].axvline(dx_best_k, color="red", ls="--")
ax[0].set_title(f"Diagnostic codes — best k = {dx_best_k}")
ax[0].set_xlabel("k"); ax[0].set_ylabel("c_v coherence")
ax[1].plot(K_VALUES, pr_scores, marker="o"); ax[1].axvline(pr_best_k, color="red", ls="--")
ax[1].set_title(f"Procedure reports — best k = {pr_best_k}")
ax[1].set_xlabel("k"); ax[1].set_ylabel("c_v coherence")
plt.tight_layout(); plt.show()
print(f"Selected topics -> diagnostic: {dx_best_k}, procedure: {pr_best_k}")

In [ ]:
def fit_lda(tfidf_corpus, dictionary, n_topics):
    return LdaModel(corpus=tfidf_corpus, id2word=dictionary, num_topics=n_topics,
                    random_state=RANDOM_STATE, passes=10, iterations=100,
                    alpha="auto", eta="auto")

lda_dx = fit_lda(dx_tfidf, dx_dict, dx_best_k)
lda_pr = fit_lda(pr_tfidf, pr_dict, pr_best_k)

def show_topics(lda, title, topn=10):
    print(f"\n=== {title} — top {topn} keywords per topic ===")
    for tid in range(lda.num_topics):
        kw = ", ".join(f"{w} ({p:.3f})" for w, p in lda.show_topic(tid, topn=topn))
        print(f"Topic {tid + 1:>2}: {kw}")

show_topics(lda_dx, "DIAGNOSTIC CODES (TF-IDF)")
show_topics(lda_pr, "PROCEDURE REPORTS (TF-IDF)")

## 8. Features: per-theme percentage contributions

Each encounter is represented by its LDA topic-distribution vector (percentage contribution of every theme) for the diagnostic and procedure models. We build three feature sets: diagnostic-only, procedure-only, and both combined.

In [ ]:
def topic_matrix(lda, corpus, n_topics):
    mat = np.zeros((len(corpus), n_topics))
    for i, doc in enumerate(corpus):
        for tid, p in lda.get_document_topics(doc, minimum_probability=0):
            mat[i, tid] = p
    return mat

X_dx = topic_matrix(lda_dx, dx_tfidf, dx_best_k)
X_pr = topic_matrix(lda_pr, pr_tfidf, pr_best_k)
y = records["los_class"].values

FEATURE_SETS = {
    "Diagnostic codes":        X_dx,
    "Procedure reports":       X_pr,
    "Both (diag + procedure)": np.hstack([X_dx, X_pr]),
}
print("combined feature shape:", FEATURE_SETS["Both (diag + procedure)"].shape)

## 9. Length-of-stay prediction

SMOTE-balanced training, 70/30 split, five classifiers (KNN k=3, Multinomial Logistic Regression, SVM, AdaBoost 100, Random Forest 100). Metrics: macro precision/recall, accuracy, and macro one-vs-rest ROC AUC.

In [ ]:
def build_classifiers():
    return {
        "kNN": KNeighborsClassifier(n_neighbors=3),
        "MLR": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "SVM": SVC(probability=True, random_state=RANDOM_STATE),
        "AC":  AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "RF":  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }

def evaluate(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
    k_neighbors = max(1, min(5, pd.Series(y_tr).value_counts().min() - 1))
    X_tr, y_tr = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors).fit_resample(X_tr, y_tr)
    classes = np.unique(y)
    y_te_bin = label_binarize(y_te, classes=classes)
    rows = {}
    for name, clf in build_classifiers().items():
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)
        try:
            auc = roc_auc_score(y_te_bin, clf.predict_proba(X_te), average="macro", multi_class="ovr")
        except Exception:
            auc = np.nan
        rows[name] = {
            "Precision": precision_score(y_te, pred, average="macro", zero_division=0),
            "Recall":    recall_score(y_te, pred, average="macro", zero_division=0),
            "Accuracy":  accuracy_score(y_te, pred),
            "ROC AUC":   auc,
        }
    return rows

results = []
for fname, X in FEATURE_SETS.items():
    for clf, metrics in evaluate(X, y).items():
        for metric, val in metrics.items():
            results.append({"Features": fname, "Classifier": clf, "Metric": metric, "Value": val})
res_df = pd.DataFrame(results)
tbl = res_df.pivot_table(index=["Features", "Metric"], columns="Classifier", values="Value")
tbl = tbl[["kNN", "MLR", "SVM", "AC", "RF"]].round(3)
tbl

In [ ]:
# Best configuration on your data
best = (res_df[res_df.Metric == "Accuracy"]
        .sort_values("Value", ascending=False).iloc[0])
print(f"Best accuracy: {best.Value:.3f}  ({best.Classifier} on '{best.Features}')")
auc_row = res_df[(res_df.Features == best.Features) &
                 (res_df.Classifier == best.Classifier) &
                 (res_df.Metric == "ROC AUC")]
if len(auc_row):
    print(f"ROC AUC for that configuration: {auc_row.Value.iloc[0]:.3f}")

## 10. Longitudinal / trend analysis (optional)

For a patient with multiple hospitalizations, trace the dominant diagnostic and procedure theme per visit to observe phenotype progression.

In [ ]:
multi = records.groupby("record_id")["encounter_id"].count().loc[lambda s: s >= 2].index

if len(multi):
    pid = multi[0]
    sub = records[records.record_id == pid].sort_values("admission")
    print(f"Patient {pid} trajectory:\n")
    for idx in sub.index:
        dd = dict(lda_dx.get_document_topics(dx_tfidf[idx], minimum_probability=0))
        pp = dict(lda_pr.get_document_topics(pr_tfidf[idx], minimum_probability=0))
        dtop = max(dd, key=dd.get) + 1
        ptop = max(pp, key=pp.get) + 1
        los = LOS_NAMES[records.loc[idx, "los_class"]]
        print(f"  {records.loc[idx,'admission'].date()} | Dx topic {dtop} | "
              f"Proc topic {ptop} | LOS: {los}")
else:
    print("No patient with multiple hospitalizations in this dataset.")